# COL-01 — Khảo sát pilot JMA trên Colab
Dùng CPU runtime mới. Upload `col01-handoff.zip` đã tạo local, nhập đường dẫn và SHA tin cậy từ máy local. Mọi thư viện chạy trong venv riêng; không mount Drive, không credential ETL, không fit mô hình. Pilot PARTIAL / DEVELOPMENT_ONLY, không suy rộng toàn Nhật Bản hoặc toàn kỳ.

In [ ]:
import hashlib, json, re, stat, subprocess, sys, uuid, zipfile
from pathlib import Path, PurePosixPath
import google.colab
assert (3, 11) <= sys.version_info[:2] <= (3, 13)
ARCHIVE = Path(input('Absolute path to col01-handoff.zip: ').strip())
EXPECTED_SHA = input('Trusted ZIP SHA256 from local receipt: ').strip()
assert re.fullmatch('[a-f0-9]{64}', EXPECTED_SHA) and hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == EXPECTED_SHA, 'COL01_ZIP_PIN_MISMATCH'
WORK = Path('/content/ie212-col01') / uuid.uuid4().hex
INPUT = WORK / 'input'
expected = {'handoff.json', 'notebooks/col_01_eda.py', 'notebooks/pilot_bundle.py', 'notebooks/requirements-col01-lock.txt', 'scripts/col01-eda.py', *('bundle/' + n for n in ('features-part-00000.parquet', 'mainshocks.parquet', 'pilot_manifest.json', 'coverage_report.json', 'checksums.sha256'))}
with zipfile.ZipFile(ARCHIVE) as z:
    members = z.infolist()
    assert len(members) == len(expected) and {m.filename for m in members} == expected
    assert sum(m.file_size for m in members) < 2*1024**2
    for m in members:
        p = PurePosixPath(m.filename)
        assert not p.is_absolute() and '..' not in p.parts and not stat.S_ISLNK(m.external_attr >> 16) and not m.flag_bits & 1
    INPUT.mkdir(parents=True, exist_ok=False)
    z.extractall(INPUT)
handoff = json.loads((INPUT / 'handoff.json').read_text())
assert handoff['version'] == 'col01-handoff-v1' and set(handoff['files']) == expected - {'handoff.json'}
for name, pin in handoff['files'].items():
    assert hashlib.sha256((INPUT / name).read_bytes()).hexdigest() == pin
print('COL01_HANDOFF_VERIFIED', EXPECTED_SHA)

In [ ]:
ENV = WORK / 'venv'
def checked(args):
    result = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode: print(result.stderr)
    result.check_returncode()
checked([sys.executable, '-m', 'venv', '--without-pip', ENV])
PYTHON = ENV / 'bin/python'
checked([sys.executable, '-m', 'pip', '--isolated', '--python', PYTHON, 'install', '--disable-pip-version-check', '-r', INPUT / 'notebooks/requirements-col01-lock.txt'])
checked([sys.executable, '-m', 'pip', '--isolated', '--python', PYTHON, 'check'])
print('COL01_ENV_READY')

In [ ]:
OUTPUT = WORK / 'output'
checked([PYTHON, '-B', INPUT / 'scripts/col01-eda.py', 'run', '--bundle', INPUT / 'bundle', '--output', OUTPUT, '--code-version', handoff['code_version'], '--runtime', 'GOOGLE_COLAB'])
report = json.loads((OUTPUT / 'pilot-eda.json').read_text())
assert report['runtime_kind'] == 'GOOGLE_COLAB' and report['input_unchanged']
assert report['feature_rows'] == 279 and report['mainshocks'] == 2
assert report['research_ready'] is False and report['allowed_usage'] == 'DEVELOPMENT_ONLY'
print('COL01_COLAB_PASSED', report['unique_candidate_ids'], report['unique_mainshock_candidate_pairs'])
print('Per-window PRE/MAINSHOCK/POST:', report['windows'])
print('Coverage meaning:', report['coverage_meaning'])
for window in report['coverage_windows']:
    print(window)
print('Scaling:', report['scaling'])
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'window-counts.png')))

## Diễn giải
279 dòng là grain mainshock/candidate; unique candidate ID phải tính độc lập, không mặc định bằng số dòng. `windows.csv` ghi PRE/MAINSHOCK/POST và self-only. Km/giờ là đơn vị vật lý; x/y/z/time scaled không có đơn vị. Coverage chỉ đo khoảng input có dữ liệu, không chứng minh catalog đầy đủ. Self-only và selection bias giới hạn diễn giải; không được gọi PRE là dư chấn. JSON lưu null/nonfinite/self vectors/scaling, input SHA, module SHA và package versions.

In [ ]:
pins = json.loads((OUTPUT / 'output-checksums.json').read_text())
OUTPUT_ZIP = WORK / 'col01-output.zip'
with zipfile.ZipFile(OUTPUT_ZIP, 'x', compression=zipfile.ZIP_DEFLATED) as z:
    for name in sorted(set(pins) | {'output-checksums.json'}):
        source = OUTPUT / name
        if name in pins: assert hashlib.sha256(source.read_bytes()).hexdigest() == pins[name]
        z.write(source, name)
with zipfile.ZipFile(OUTPUT_ZIP) as z:
    for name, pin in pins.items(): assert hashlib.sha256(z.read(name)).hexdigest() == pin
print('COL01_OUTPUT_ZIP_VERIFIED', hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest())
from google.colab import files
files.download(str(OUTPUT_ZIP))